# Training semantic segmentation (DeepLabv3+ or PSPNet) in Google Colab with CVAT masks and Azure Blob storage

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/datasantana/sam-image-processing/blob/main/notebooks/semantic_segmentation_training.ipynb)

In [1]:
# 🔧 Install Dependencies
# Run this cell first when opening the notebook in Google Colab

!pip install azure-storage-blob>=12.17.0
!pip install opencv-python>=4.8.0
!pip install pillow>=10.0.0
!pip install numpy>=1.24.0
!pip install matplotlib>=3.7.0

print("🎉 Dependencies installed successfully!")

🎉 Dependencies installed successfully!


# 🛰️ Semantic Segmentation Training Setup

This notebook initializes the connection to Azure Blob Storage, downloads drone imagery and labeled masks, and prepares reproducible training and validation datasets for semantic segmentation using DeepLabv3+ or PSPNet.

**Dataset structure:**
- Images: `assets/drone-imagery`
- Masks: `assets/pascual_annotation_masks/SegmentationClass`
- Label map: `assets/pascual_annotation_masks/labelmap.txt`

**Azure Blob Storage:**
- Container: `general-purpose`
- Connection string: stored securely in `AZURE_CONN_STR`

## 1: Azure Blob Connection & Download

In [2]:
# 📦 Connect to Azure Blob Storage and download images and masks
from azure.storage.blob import BlobServiceClient
from google.colab import userdata
import os

# 🔐 Azure Connection Configuration
# Option 1: Set as environment variable (recommended for Colab)
# import os
AZURE_CONN_STR = userdata.get('AZURE_CONNECTION_STRING')

# Option 2: Use getpass for secure input (uncomment to use)
# from getpass import getpass
# AZURE_CONN_STR = getpass('Enter Azure Connection String: ')

# Option 3: Set directly (replace with your actual connection string)
# AZURE_CONN_STR = "DefaultEndpointsProtocol=...<place complete string here>"

if AZURE_CONN_STR == "DefaultEndpointsProtocol=...<place complete string here>":
    print("⚠️  Please set your Azure connection string!")
    print("Uncomment one of the options above or set AZURE_CONN_STR variable")
    raise ValueError("Azure connection string not configured")

CONTAINER_NAME = "general-purpose"

# Blob Storage paths (where data is stored in Azure)
LOCAL_IMG_DIR = "assets/drone-imagery"
LOCAL_MASK_DIR = "assets/pascual_annotation_masks/SegmentationClass"
LOCAL_LABELMAP_DIR = "assets/pascual_annotation_masks"

# Local directories (where to download files)
LOCAL_DOWNLOAD_IMG_DIR = "assets/images"
LOCAL_DOWNLOAD_MASK_DIR = "assets/masks"

os.makedirs(LOCAL_DOWNLOAD_IMG_DIR, exist_ok=True)
os.makedirs(LOCAL_DOWNLOAD_MASK_DIR, exist_ok=True)

# Connect to blob container
try:
    blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
    container_client = blob_service_client.get_container_client(CONTAINER_NAME)
    print("✅ Successfully connected to Azure Blob Storage")
except Exception as e:
    print(f"❌ Failed to connect to Azure Blob Storage: {e}")
    raise

def download_blobs(prefix, local_dir, file_types=None):
    """
    Download blobs with optional file type filtering

    Args:
        prefix: Blob prefix to search for
        local_dir: Local directory to download to
        file_types: List of extensions (case-insensitive) or None to download all files
                   e.g., ['.jpg', '.jpeg', '.png'] or None
    """
    print(f"🔍 Looking for blobs with prefix '{prefix}'...")
    if file_types:
        print(f"   File types: {file_types} (case-insensitive)")
    else:
        print("   Downloading all file types")

    try:
        blobs = list(container_client.list_blobs(name_starts_with=prefix))
        print(f"📁 Found {len(blobs)} total blobs with prefix '{prefix}'")

        downloaded_count = 0
        for blob in blobs:
            print(f"  - Checking blob: {blob.name}")

            # Check file type if specified
            should_download = True
            if file_types:
                should_download = any(blob.name.lower().endswith(ext.lower()) for ext in file_types)

            if should_download:
                local_path = os.path.join(local_dir, os.path.basename(blob.name))
                if not os.path.exists(local_path):
                    print(f"    ⬇️ Downloading {blob.name} -> {local_path}")
                    try:
                        with open(local_path, "wb") as f:
                            blob_data = container_client.download_blob(blob.name).readall()
                            f.write(blob_data)
                        downloaded_count += 1
                        print(f"    ✅ Downloaded {blob.name} ({len(blob_data)} bytes)")
                    except Exception as e:
                        print(f"    ❌ Failed to download {blob.name}: {e}")
                else:
                    print(f"    ⏭️ Skipping {blob.name} (already exists)")
            else:
                print(f"    ⏭️ Skipping {blob.name} (file type not in {file_types})")

        print(f"📊 Downloaded {downloaded_count} files from {prefix} to {local_dir}")

    except Exception as e:
        print(f"❌ Error listing blobs with prefix '{prefix}': {e}")

# Download images and masks with flexible file type matching
# Images: Support common image formats (case-insensitive)
download_blobs(LOCAL_IMG_DIR, LOCAL_DOWNLOAD_IMG_DIR, ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'])

# Masks: Support common mask formats (case-insensitive)
download_blobs(LOCAL_MASK_DIR, LOCAL_DOWNLOAD_MASK_DIR, ['.png', '.jpg', '.jpeg', '.bmp'])

✅ Successfully connected to Azure Blob Storage
🔍 Looking for blobs with prefix 'assets/drone-imagery'...
   File types: ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'] (case-insensitive)
📁 Found 66 total blobs with prefix 'assets/drone-imagery'
  - Checking blob: assets/drone-imagery/DJI_20250522111442_0006_D.JPG
    ⬇️ Downloading assets/drone-imagery/DJI_20250522111442_0006_D.JPG -> assets/images/DJI_20250522111442_0006_D.JPG
    ✅ Downloaded assets/drone-imagery/DJI_20250522111442_0006_D.JPG (7315456 bytes)
  - Checking blob: assets/drone-imagery/DJI_20250522111502_0016_D.JPG
    ⬇️ Downloading assets/drone-imagery/DJI_20250522111502_0016_D.JPG -> assets/images/DJI_20250522111502_0016_D.JPG
    ✅ Downloaded assets/drone-imagery/DJI_20250522111502_0016_D.JPG (7458816 bytes)
  - Checking blob: assets/drone-imagery/DJI_20250522111506_0018_D.JPG
    ⬇️ Downloading assets/drone-imagery/DJI_20250522111506_0018_D.JPG -> assets/images/DJI_20250522111506_0018_D.JPG
    ✅ Downloaded assets

## 2: 🧪 Dataset Preparation

This section reads and sorts the downloaded images and masks, then creates a reproducible split:
- First 53 images → Training set
- Remaining images → Validation set

Each image is matched with its corresponding mask by filename.

In [3]:
# List and sort all image and mask files (no format validation)
image_files = sorted([f for f in os.listdir(LOCAL_DOWNLOAD_IMG_DIR)])
mask_files = sorted([f for f in os.listdir(LOCAL_DOWNLOAD_MASK_DIR)])

print(f"📁 Found {len(image_files)} image files")
print(f"📁 Found {len(mask_files)} mask files")

# Match image-mask pairs by base filename (remove extensions)
def get_base_name(filename):
    return os.path.splitext(filename)[0]

# Create pairs based on matching base names
image_base_names = {get_base_name(f): f for f in image_files}
mask_base_names = {get_base_name(f): f for f in mask_files}

# Find matching pairs
matching_base_names = set(image_base_names.keys()) & set(mask_base_names.keys())
pairs = [(image_base_names[base], mask_base_names[base]) for base in sorted(matching_base_names)]

print(f"🔗 Found {len(pairs)} matching image-mask pairs")

# Reproducible split: first 53 for training
train_pairs = pairs[:53]
val_pairs = pairs[53:]

# Build full path dictionaries
def build_dataset(pairs, img_dir, mask_dir):
    return [
        {
            "img_path": os.path.join(img_dir, img),
            "seg_map_path": os.path.join(mask_dir, mask)
        }
        for img, mask in pairs
    ]

train_dataset = build_dataset(train_pairs, LOCAL_DOWNLOAD_IMG_DIR, LOCAL_DOWNLOAD_MASK_DIR)
val_dataset = build_dataset(val_pairs, LOCAL_DOWNLOAD_IMG_DIR, LOCAL_DOWNLOAD_MASK_DIR)

print(f"✅ Train set: {len(train_dataset)} images")
print(f"✅ Val set: {len(val_dataset)} images")
print(f"📁 Images directory: {LOCAL_DOWNLOAD_IMG_DIR}")
print(f"📁 Masks directory: {LOCAL_DOWNLOAD_MASK_DIR}")
print(f"🔗 Blob source - Images: {LOCAL_IMG_DIR}")
print(f"🔗 Blob source - Masks: {LOCAL_MASK_DIR}")

📁 Found 66 image files
📁 Found 66 mask files
🔗 Found 66 matching image-mask pairs
✅ Train set: 53 images
✅ Val set: 13 images
📁 Images directory: assets/images
📁 Masks directory: assets/masks
🔗 Blob source - Images: assets/drone-imagery
🔗 Blob source - Masks: assets/pascual_annotation_masks/SegmentationClass


## 3: 🚀 Model Training Setup

Now we'll set up the semantic segmentation training pipeline with support for:
- **DeepLabv3+** with ResNet backbone
- **PSPNet** with ResNet backbone  
- Easy model switching
- Configurable training parameters
- Label map integration

### 3.1: 🔧 Deep Learning Environment Setup

In [4]:
# 🔧 Install Deep Learning Dependencies
# PyTorch and related packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install segmentation-models-pytorch>=0.3.3
!pip install albumentations>=1.3.0
!pip install timm>=0.9.0

# Additional ML utilities
!pip install scikit-learn>=1.3.0
!pip install tqdm>=4.65.0
!pip install tensorboard>=2.13.0

print("🎉 Deep learning environment setup complete!")

Looking in indexes: https://download.pytorch.org/whl/cu118
🎉 Deep learning environment setup complete!


### 3.2: 📋 Model Configuration

In [5]:
# 📋 Model and Training Configuration
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 🎯 Model Selection - Change this to switch models
MODEL_ARCH = "DeepLabv3+"  # Options: "DeepLabv3+", "PSPNet"
BACKBONE = "resnet50"      # Options: "resnet18", "resnet34", "resnet50", "resnet101", "resnet152"

# 🏋️ Training Configuration
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
NUM_EPOCHS = 50
IMAGE_SIZE = 512
NUM_WORKERS = 2

# 📊 Model Architecture Settings
if MODEL_ARCH == "DeepLabv3+":
    print(f"🎯 Using DeepLabv3+ with {BACKBONE} backbone")
    model_class = smp.DeepLabV3Plus
elif MODEL_ARCH == "PSPNet":
    print(f"🎯 Using PSPNet with {BACKBONE} backbone")
    model_class = smp.PSPNet
else:
    raise ValueError(f"Unsupported model architecture: {MODEL_ARCH}")

# 🖥️ Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

# 📁 Output Configuration
MODEL_SAVE_DIR = "models"
LOGS_DIR = "logs"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

print(f"✅ Configuration complete!")
print(f"   Model: {MODEL_ARCH}")
print(f"   Backbone: {BACKBONE}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Image Size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"   Epochs: {NUM_EPOCHS}")

🎯 Using DeepLabv3+ with resnet50 backbone
🖥️ Using device: cuda
✅ Configuration complete!
   Model: DeepLabv3+
   Backbone: resnet50
   Batch Size: 8
   Learning Rate: 0.0001
   Image Size: 512x512
   Epochs: 50


### 3.3: 🏷️ Class Labels and Label Map

In [6]:
# 🏷️ Load and Process Label Map
import numpy as np
from PIL import Image
import json
import cv2

# Local path where labelmap will be downloaded
local_labelmap_path = "assets/labelmap.txt"

try:
    # Try to download labelmap from blob storage
    print("🔍 Looking for labelmap in blob storage...")
    labelmap_blobs = list(container_client.list_blobs(name_starts_with=LOCAL_LABELMAP_DIR))

    labelmap_found = False
    for blob in labelmap_blobs:
        if 'labelmap' in blob.name.lower() and blob.name.endswith('.txt'):
            print(f"📥 Downloading labelmap: {blob.name} -> {local_labelmap_path}")
            with open(local_labelmap_path, "wb") as f:
                f.write(container_client.download_blob(blob.name).readall())
            labelmap_found = True
            break

    if not labelmap_found:
        raise FileNotFoundError("No labelmap.txt file found in blob storage")

    # Load labelmap from downloaded file
    if os.path.exists(local_labelmap_path):
        with open(local_labelmap_path, 'r') as f:
            class_names = [line.strip() for line in f.readlines() if line.strip()]
        print(f"✅ Loaded {len(class_names)} classes from labelmap")
    else:
        raise FileNotFoundError("Labelmap download failed")

except Exception as e:
    print(f"⚠️ Could not load labelmap: {e}")
    print("🔧 Creating default labelmap by analyzing mask files...")

    # Check if mask variables are available
    if 'LOCAL_DOWNLOAD_MASK_DIR' in locals() and 'mask_files' in locals():
        # Analyze a few mask files to determine unique classes
        sample_masks = mask_files[:min(10, len(mask_files))]
        unique_values = set()

        for mask_file in sample_masks:
            mask_path = os.path.join(LOCAL_DOWNLOAD_MASK_DIR, mask_file)
            try:
                mask = np.array(Image.open(mask_path))
                unique_values.update(np.unique(mask))
            except Exception as mask_error:
                print(f"⚠️ Could not read mask {mask_file}: {mask_error}")

        # Create class names
        unique_values = sorted(list(unique_values))
        class_names = [f"class_{i}" for i in unique_values]
        print(f"🔍 Found {len(unique_values)} unique classes: {unique_values}")
    else:
        print("⚠️ Mask files not available. Creating minimal class set.")
        class_names = ['background', 'foreground']

# Set up classes
NUM_CLASSES = len(class_names)
print(f"📊 Number of classes: {NUM_CLASSES}")
print(f"🏷️ Class names: {class_names}")

# Create class to index mapping
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
idx_to_class = {idx: name for idx, name in enumerate(class_names)}

# 🎨 RGB Mask Detection and Processing
print("\n🔍 Analyzing mask format...")

# Initialize RGB to class mapping
rgb_to_class = None

# Check if mask data is available for RGB analysis
if 'LOCAL_DOWNLOAD_MASK_DIR' in locals() and 'mask_files' in locals() and len(mask_files) > 0:
    # Check if masks are RGB or grayscale by examining a sample
    sample_mask_path = os.path.join(LOCAL_DOWNLOAD_MASK_DIR, mask_files[0])

    try:
        sample_mask = cv2.imread(sample_mask_path)
        sample_mask_gray = cv2.imread(sample_mask_path, cv2.IMREAD_GRAYSCALE)

        # Detect if mask is RGB (color) or grayscale
        is_rgb_mask = False
        if sample_mask is not None and len(sample_mask.shape) == 3:
            # Check if all channels are different (RGB) or same (grayscale saved as 3-channel)
            if not np.array_equal(sample_mask[:,:,0], sample_mask[:,:,1]) or not np.array_equal(sample_mask[:,:,1], sample_mask[:,:,2]):
                is_rgb_mask = True

        print(f"🎨 Mask format detected: {'RGB Color' if is_rgb_mask else 'Grayscale'}")

        if is_rgb_mask:
            print("🎨 Processing RGB masks...")

            # Try to parse RGB values from labelmap if available
            if os.path.exists(local_labelmap_path):
                print("📋 Parsing RGB values from labelmap...")
                rgb_to_class = {}

                try:
                    with open(local_labelmap_path, 'r') as f:
                        lines = f.readlines()

                    for idx, line in enumerate(lines):
                        line = line.strip()
                        if ':' in line and ',' in line:
                            # Format: "ClassName:R,G,B"
                            parts = line.split(':')
                            if len(parts) >= 2:
                                class_name = parts[0].strip()
                                rgb_str = parts[1].strip()
                                try:
                                    r, g, b = map(int, rgb_str.split(','))
                                    # Convert RGB to BGR for OpenCV compatibility
                                    bgr_color = (b, g, r)
                                    rgb_to_class[bgr_color] = idx
                                    print(f"   {class_name}: RGB({r},{g},{b}) -> BGR{bgr_color} -> Class {idx}")
                                except ValueError as ve:
                                    print(f"   ⚠️ Could not parse RGB values for: {line} - {ve}")

                    print(f"✅ Created RGB-to-class mapping with {len(rgb_to_class)} colors")

                except Exception as e:
                    print(f"⚠️ Error parsing RGB labelmap: {e}")
                    rgb_to_class = None

            if rgb_to_class is None or len(rgb_to_class) == 0:
                print("🔍 Analyzing RGB colors in masks...")
                # Analyze actual mask files to build color mapping
                unique_colors = set()

                for mask_file in mask_files[:min(5, len(mask_files))]:
                    mask_path = os.path.join(LOCAL_DOWNLOAD_MASK_DIR, mask_file)
                    try:
                        mask = cv2.imread(mask_path)
                        if mask is not None:
                            # Get unique BGR colors
                            reshaped = mask.reshape(-1, 3)
                            unique_colors.update([tuple(color) for color in reshaped])
                    except Exception as e:
                        print(f"   ⚠️ Could not analyze {mask_file}: {e}")

                # Create mapping from unique colors to class indices
                unique_colors = sorted(list(unique_colors))[:NUM_CLASSES]  # Limit to expected number of classes
                rgb_to_class = {color: idx for idx, color in enumerate(unique_colors)}

                print(f"🎨 Found {len(unique_colors)} unique colors in masks:")
                for color, idx in list(rgb_to_class.items())[:5]:
                    print(f"   BGR{color} -> Class {idx}")
                if len(rgb_to_class) > 5:
                    print(f"   ... and {len(rgb_to_class) - 5} more")
        else:
            print("📊 Masks are grayscale - no RGB mapping needed")

    except Exception as e:
        print(f"⚠️ Error analyzing mask format: {e}")
        print("📊 Defaulting to grayscale mode")

else:
    print("⚠️ Mask files not available for RGB analysis")
    print("📊 Run the data download cell first, then re-run this cell")

print(f"\n✅ Label mapping complete!")
print(f"   Classes: {NUM_CLASSES}")
print(f"   Mapping: {dict(list(class_to_idx.items())[:5])}{'...' if len(class_to_idx) > 5 else ''}")
print(f"   RGB Support: {'Enabled' if rgb_to_class and len(rgb_to_class) > 0 else 'Disabled'}")
print(f"🔗 Blob source - Labelmap: {LOCAL_LABELMAP_DIR}")
print(f"📁 Local labelmap: {local_labelmap_path if os.path.exists(local_labelmap_path) else 'Generated from masks'}")

# Debug info
if rgb_to_class:
    print(f"🎨 RGB mapping contains {len(rgb_to_class)} color mappings")
    sample_colors = list(rgb_to_class.items())[:3]
    for color, class_idx in sample_colors:
        print(f"   Color {color} -> Class {class_idx}")
else:
    print("🎨 No RGB mapping created - check that masks are RGB and data is downloaded")

🔍 Looking for labelmap in blob storage...
📥 Downloading labelmap: assets/pascual_annotation_masks/labelmap.txt -> assets/labelmap.txt
✅ Loaded 14 classes from labelmap
📊 Number of classes: 14
🏷️ Class names: ['# label:color_rgb:parts:actions', 'Brownfields:178,80,80::', 'Buildings:250,50,83::', 'Fence:184,61,245::', 'Ground Surface:255,204,51::', 'Lightning Posts:245,147,49::', 'Parkinglots:140,120,240::', 'Paved Surface:52,209,183::', 'Pools:51,221,255::', 'Road Surface:89,134,179::', 'Sports Fields:250,250,55::', 'Vegetation:36,179,83::', 'Vehicle:204,51,102::', 'background:0,0,0::']

🔍 Analyzing mask format...
🎨 Mask format detected: RGB Color
🎨 Processing RGB masks...
📋 Parsing RGB values from labelmap...
   Brownfields: RGB(178,80,80) -> BGR(80, 80, 178) -> Class 1
   Buildings: RGB(250,50,83) -> BGR(83, 50, 250) -> Class 2
   Fence: RGB(184,61,245) -> BGR(245, 61, 184) -> Class 3
   Ground Surface: RGB(255,204,51) -> BGR(51, 204, 255) -> Class 4
   Lightning Posts: RGB(245,147,49

### 3.4: 🔄 Data Transforms and Dataset

In [7]:
# 🔄 RGB-Aware Dataset and Transforms
from torch.utils.data import Dataset
import cv2

class RGBSegmentationDataset(Dataset):
    def __init__(self, dataset, transforms=None, rgb_to_class=None, use_rgb=False):
        self.dataset = dataset
        self.transforms = transforms
        self.rgb_to_class = rgb_to_class
        self.use_rgb = use_rgb

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]

        # Load image
        image = cv2.imread(item['img_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.use_rgb and self.rgb_to_class is not None:
            # Load mask as RGB and convert to class indices
            mask_rgb = cv2.imread(item['seg_map_path'], cv2.IMREAD_COLOR)
            mask = self.rgb_to_class_indices(mask_rgb)
        else:
            # Load mask as grayscale (fallback)
            mask = cv2.imread(item['seg_map_path'], cv2.IMREAD_GRAYSCALE)

        # Apply transforms
        if self.transforms:
            transformed = self.transforms(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']

        return image, mask.long()

    def rgb_to_class_indices(self, rgb_mask):
        """Convert RGB mask to class indices"""
        height, width, _ = rgb_mask.shape
        class_mask = np.zeros((height, width), dtype=np.uint8)

        for y in range(height):
            for x in range(width):
                pixel_bgr = tuple(rgb_mask[y, x])

                if pixel_bgr in self.rgb_to_class:
                    class_mask[y, x] = self.rgb_to_class[pixel_bgr]
                else:
                    # Find closest color if exact match not found
                    min_distance = float('inf')
                    closest_class = 0

                    for color, class_idx in self.rgb_to_class.items():
                        distance = np.sqrt(sum((pixel_bgr[i] - color[i])**2 for i in range(3)))
                        if distance < min_distance:
                            min_distance = distance
                            closest_class = class_idx

                    class_mask[y, x] = closest_class

        return class_mask

# 🎨 Define Transforms
train_transforms = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.RandomGamma(p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# 📊 Create Datasets with RGB Support
print("🔄 Creating datasets...")

# Check if we have RGB mask support (safely check for rgb_to_class)
try:
    use_rgb_masks = rgb_to_class is not None and len(rgb_to_class) > 0
except NameError:
    use_rgb_masks = False
    rgb_to_class = None

if use_rgb_masks:
    print("🎨 Using RGB mask support...")
    train_ds = RGBSegmentationDataset(train_dataset, transforms=train_transforms,
                                     rgb_to_class=rgb_to_class, use_rgb=True)
    val_ds = RGBSegmentationDataset(val_dataset, transforms=val_transforms,
                                   rgb_to_class=rgb_to_class, use_rgb=True)
else:
    print("📊 Using grayscale mask fallback...")
    train_ds = RGBSegmentationDataset(train_dataset, transforms=train_transforms,
                                     rgb_to_class=None, use_rgb=False)
    val_ds = RGBSegmentationDataset(val_dataset, transforms=val_transforms,
                                   rgb_to_class=None, use_rgb=False)

# 🔄 Create DataLoaders
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"✅ Datasets created successfully!")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Samples per batch: {BATCH_SIZE}")
print(f"   RGB mask support: {'Enabled' if use_rgb_masks else 'Disabled'}")

# 🧪 Test data loading
try:
    sample_image, sample_mask = next(iter(train_loader))
    print(f"📊 Data loading test successful!")
    print(f"   Image shape: {sample_image.shape}")
    print(f"   Mask shape: {sample_mask.shape}")
    print(f"   Image range: [{sample_image.min():.3f}, {sample_image.max():.3f}]")
    print(f"   Mask classes: {torch.unique(sample_mask)}")
    print(f"   Mask range: [{sample_mask.min()}, {sample_mask.max()}]")

    # Validate mask values are in correct range
    if sample_mask.max() < NUM_CLASSES and sample_mask.min() >= 0:
        print(f"✅ Mask values in valid range [0, {NUM_CLASSES-1}]")
    else:
        print(f"❌ Invalid mask values! Range: [{sample_mask.min()}, {sample_mask.max()}], Expected: [0, {NUM_CLASSES-1}]")

except Exception as e:
    print(f"❌ Data loading test failed: {e}")

🔄 Creating datasets...
🎨 Using RGB mask support...
✅ Datasets created successfully!
   Training batches: 7
   Validation batches: 2
   Samples per batch: 8
   RGB mask support: Enabled
📊 Data loading test successful!
   Image shape: torch.Size([8, 3, 512, 512])
   Mask shape: torch.Size([8, 512, 512])
   Image range: [-2.118, 2.640]
   Mask classes: tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13])
   Mask range: [1, 13]
✅ Mask values in valid range [0, 13]


### 3.5: 🏗️ Model Initialization

This cell creates the PyTorch model and sets up training components:

- **Model Creation**: DeepLabv3+ or PSPNet with ResNet backbone
- **Loss Function**: DiceLoss for multiclass segmentation  
- **Optimizer**: Adam with weight decay
- **Scheduler**: ReduceLROnPlateau for adaptive learning rate

In [8]:
# 🏗️ Model Initialization
print("🏗️ Initializing model...")

# Create the model
print(f"🎯 Creating {MODEL_ARCH} with {BACKBONE} backbone...")
model = model_class(
    encoder_name=BACKBONE,
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
)

# Move model to device
model = model.to(device)

# Print model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model created successfully!")
print(f"   Architecture: {MODEL_ARCH}")
print(f"   Backbone: {BACKBONE}")
print(f"   Input channels: 3")
print(f"   Output classes: {NUM_CLASSES}")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Device: {device}")

# 🎯 Loss Function and Optimizer Setup
print("\n⚖️ Setting up loss function and optimizer...")

# Loss function - DiceLoss works well for segmentation
criterion = smp.losses.DiceLoss(mode='multiclass')

# Optimizer - Adam with weight decay
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Learning rate scheduler - reduces LR when validation loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

print(f"✅ Training setup complete!")
print(f"   Loss function: DiceLoss (multiclass)")
print(f"   Optimizer: Adam (lr={LEARNING_RATE})")
print(f"   Scheduler: ReduceLROnPlateau")
print(f"   Weight decay: 1e-4")

🏗️ Initializing model...
🎯 Creating DeepLabv3+ with resnet50 backbone...


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

✅ Model created successfully!
   Architecture: DeepLabv3+
   Backbone: resnet50
   Input channels: 3
   Output classes: 14
   Total parameters: 26,680,926
   Trainable parameters: 26,680,926
   Device: cuda

⚖️ Setting up loss function and optimizer...
✅ Training setup complete!
   Loss function: DiceLoss (multiclass)
   Optimizer: Adam (lr=0.0001)
   Scheduler: ReduceLROnPlateau
   Weight decay: 1e-4


### 3.6: 🧪 Forward Pass Test

**What is a Forward Pass Test?**
- Tests that the model can process data without errors
- Verifies input/output shapes are correct (e.g., [batch, 3, 512, 512] → [batch, 14, 512, 512])
- Catches issues like BatchNorm errors before training starts
- Uses `model.eval()` mode to avoid BatchNorm issues with batch size = 1

**Why is it needed?**
- Saves time by catching configuration errors early
- Validates that NUM_CLASSES matches model output
- Ensures GPU memory is sufficient for the model
- Prevents crashes during actual training

In [9]:
# 🧪 Fixed Forward Pass Test
# The BatchNorm error occurs because the model is in training mode with batch_size=1
# Solution: Set model to eval() mode for testing

print("🧪 Testing model forward pass...")

try:
    # Set model to evaluation mode to avoid BatchNorm issues
    model.eval()

    with torch.no_grad():
        # Test with different batch sizes
        test_batch_sizes = [1, 2]

        for batch_size in test_batch_sizes:
            dummy_input = torch.randn(batch_size, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
            dummy_output = model(dummy_input)

            print(f"✅ Batch size {batch_size} test successful!")
            print(f"   Input shape: {dummy_input.shape}")
            print(f"   Output shape: {dummy_output.shape}")
            print(f"   Output classes: {dummy_output.shape[1]} (expected: {NUM_CLASSES})")

    # Set back to training mode for actual training
    model.train()
    print(f"🔄 Model set back to training mode")

except Exception as e:
    print(f"❌ Forward pass test failed: {e}")
    print("💡 This might indicate a model architecture issue")

print(f"\n🎯 Model is ready for training!")
print(f"   Architecture: {MODEL_ARCH}")
print(f"   Backbone: {BACKBONE}")
print(f"   Classes: {NUM_CLASSES}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

🧪 Testing model forward pass...
✅ Batch size 1 test successful!
   Input shape: torch.Size([1, 3, 512, 512])
   Output shape: torch.Size([1, 14, 512, 512])
   Output classes: 14 (expected: 14)
✅ Batch size 2 test successful!
   Input shape: torch.Size([2, 3, 512, 512])
   Output shape: torch.Size([2, 14, 512, 512])
   Output classes: 14 (expected: 14)
🔄 Model set back to training mode

🎯 Model is ready for training!
   Architecture: DeepLabv3+
   Backbone: resnet50
   Classes: 14
   Parameters: 26,680,926


## 4: 🚀 Training Loop

**Current Status**: We have a **READY-TO-TRAIN** model, not a trained one yet!

So far we've only:
- ✅ Downloaded data from Azure Blob Storage
- ✅ Prepared datasets and data loaders  
- ✅ Initialized the model architecture
- ✅ Set up loss function, optimizer, and scheduler

**Next**: Run the actual training to get a **TRAINED** model!

In [10]:
# 🏃‍♂️ ACTUAL TRAINING LOOP - This creates the TRAINED model!
print("🚀 Starting training...")
print(f"   Model: {MODEL_ARCH} with {BACKBONE} backbone")
print(f"   Dataset: {len(train_dataset)} train, {len(val_dataset)} val samples")
print(f"   Training for {NUM_EPOCHS} epochs")
print("="*60)

# Training history
train_losses = []
train_ious = []
val_losses = []
val_ious = []
best_val_iou = 0.0

# Start training
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    print(f"\n📅 Epoch {epoch+1}/{NUM_EPOCHS}")

    # Train
    train_loss, train_iou = train_one_epoch(
        model, train_loader, criterion, optimizer, device, epoch
    )

    # Validate
    val_loss, val_iou = validate_one_epoch(
        model, val_loader, criterion, device
    )

    # Update learning rate
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    # Save metrics
    train_losses.append(train_loss)
    train_ious.append(train_iou)
    val_losses.append(val_loss)
    val_ious.append(val_iou)

    # Print epoch summary
    print(f"📊 Epoch {epoch+1} Results:")
    print(f"   Train Loss: {train_loss:.4f} | Train IoU: {train_iou:.4f}")
    print(f"   Val Loss: {val_loss:.4f} | Val IoU: {val_iou:.4f}")
    print(f"   Learning Rate: {current_lr:.2e}")

    # Save best model
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        best_model_path = f"{MODEL_SAVE_DIR}/best_{MODEL_ARCH}_{BACKBONE}_epoch_{epoch+1}.pth"
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_iou': val_iou,
            'model_arch': MODEL_ARCH,
            'backbone': BACKBONE,
            'num_classes': NUM_CLASSES,
            'class_names': class_names
        }, best_model_path)
        print(f"💾 New best model saved! IoU: {val_iou:.4f}")

# Training completed
end_time = time.time()
training_time = end_time - start_time

print("="*60)
print("🎉 TRAINING COMPLETED!")
print(f"⏱️ Total training time: {training_time/3600:.2f} hours")
print(f"🏆 Best validation IoU: {best_val_iou:.4f}")
print(f"💾 Best model saved to: {best_model_path}")
print(f"📊 Training history: {len(train_losses)} epochs")

# Final model save
final_model_path = f"{MODEL_SAVE_DIR}/final_{MODEL_ARCH}_{BACKBONE}.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'training_history': {
        'train_losses': train_losses,
        'train_ious': train_ious,
        'val_losses': val_losses,
        'val_ious': val_ious
    },
    'model_arch': MODEL_ARCH,
    'backbone': BACKBONE,
    'num_classes': NUM_CLASSES,
    'class_names': class_names
}, final_model_path)

print(f"💾 Final model saved to: {final_model_path}")
print("\n🎯 YOU NOW HAVE A TRAINED MODEL! 🎯")

🚀 Starting training...
   Model: DeepLabv3+ with resnet50 backbone
   Dataset: 53 train, 13 val samples
   Training for 50 epochs


NameError: name 'time' is not defined

### 4.3: 📈 Training Results Visualization

In [ ]:
# 📈 Plot Training Results
import matplotlib.pyplot as plt

# Create training plots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Loss plots
epochs = range(1, len(train_losses) + 1)

ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
ax1.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
ax1.set_title('Training vs Validation Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# IoU plots
ax2.plot(epochs, train_ious, 'b-', label='Training IoU', linewidth=2)
ax2.plot(epochs, val_ious, 'r-', label='Validation IoU', linewidth=2)
ax2.set_title('Training vs Validation IoU')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('IoU')
ax2.legend()
ax2.grid(True)

# Learning rate plot
ax3.plot(epochs, [optimizer.param_groups[0]['lr']] * len(epochs), 'g-', linewidth=2)
ax3.set_title('Learning Rate Schedule')
ax3.set_xlabel('Epochs')
ax3.set_ylabel('Learning Rate')
ax3.set_yscale('log')
ax3.grid(True)

# Best metrics summary
metrics_text = f"""
Final Training Results:
━━━━━━━━━━━━━━━━━━━━━━

🏆 Best Validation IoU: {best_val_iou:.4f}
📉 Final Training Loss: {train_losses[-1]:.4f}
📉 Final Validation Loss: {val_losses[-1]:.4f}
📊 Final Training IoU: {train_ious[-1]:.4f}
📊 Final Validation IoU: {val_ious[-1]:.4f}

🔧 Model: {MODEL_ARCH}
🔧 Backbone: {BACKBONE}
🔧 Classes: {NUM_CLASSES}
⏱️ Training Time: {training_time/3600:.2f} hours
"""

ax4.text(0.1, 0.5, metrics_text, transform=ax4.transAxes, fontsize=12,
         verticalalignment='center', fontfamily='monospace')
ax4.set_xlim(0, 1)
ax4.set_ylim(0, 1)
ax4.axis('off')

plt.tight_layout()
plt.savefig(f'{LOGS_DIR}/training_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Training visualization complete!")
print(f"📊 Plot saved to: {LOGS_DIR}/training_results.png")

# Print final summary
print("\n" + "="*60)
print("🎯 TRAINING SUMMARY")
print("="*60)
print(f"✅ Status: TRAINED MODEL READY!")
print(f"🏗️ Architecture: {MODEL_ARCH} with {BACKBONE}")
print(f"📊 Classes: {NUM_CLASSES}")
print(f"🏆 Best IoU: {best_val_iou:.4f}")
print(f"💾 Best model: {best_model_path}")
print(f"💾 Final model: {final_model_path}")
print("="*60)